# YOLO26 fine-tune on Colab — Roboflow 서버 직접 다운로드 (Drive 불필요)

Phase 1: 정지 차량 감지용 YOLO(car 단일 클래스) fine-tune.

**Drive 마운트 없음. 이미지 업로드 없음.** Roboflow API로 데이터셋을 Colab 서버에 바로 내려받는다 (로컬→Colab 업로드 구간 제거).

사용 순서:
1. **런타임 → 런타임 유형 변경 → GPU (T4/L4)**
2. Roboflow 웹에서 jpg 업로드 + car bbox 라벨링 후, **Generate → Export → YOLOv8 포맷 → `Show download code`** (YOLO26도 동일 YOLO 라벨 포맷) 에서 workspace/project/version/api_key 확인
3. 아래 `CONFIG` 셀에 그 값들 채우고 위에서부터 실행
4. 끝나면 `best.pt`를 다운로드해서 `extract_labels.py --yolo_weights <best.pt>` 로 사용

## 1. 설치

In [ ]:
!pip install -q -U ultralytics roboflow   # YOLO26 지원 위해 최신 ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. CONFIG — 여기만 손대면 됨

Roboflow 프로젝트의 **Export → Show download code** 에서 아래 4개 값 확인:
- `RF_API_KEY`, `RF_WORKSPACE`, `RF_PROJECT`, `RF_VERSION`

Jetson 배포 입력 크기와 맞추려고 `IMGSZ=320`.

In [ ]:
RF_API_KEY   = 'YOUR_API_KEY'        # ← Roboflow Settings → API key
RF_WORKSPACE = 'your-workspace'      # ← download code의 workspace 슬러그
RF_PROJECT   = 'your-project'        # ← project 슬러그
RF_VERSION   = 1                     # ← version 번호

MODEL    = 'yolo26n.pt'   # YOLO26: n / s / m / l / x (NMS-free, edge 최적화)
EPOCHS   = 100
IMGSZ    = 320            # Jetson 배포 입력 크기와 맞춤
BATCH    = 32             # T4 16GB 기준. OOM이면 16
PATIENCE = 30             # early-stop
RUN_NAME = 'car_v1'
OUTPUT_DIR = '/content/yolo_runs'

## 3. Roboflow에서 데이터셋 직접 다운로드 (Drive 안 거침)

In [ ]:
import os, yaml
from roboflow import Roboflow

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download('yolov8')  # /content 아래로 다운로드

DATA_YAML = os.path.join(dataset.location, 'data.yaml')
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
print('location:', dataset.location)
print('nc   :', cfg.get('nc'))
print('names:', cfg.get('names'))   # car 단일 클래스 기대

## 4. 학습

폐쇄 환경 + 단일 클래스라 mosaic는 약하게, flip은 끔(방향 정보 보존).
결과는 `OUTPUT_DIR/RUN_NAME/weights/best.pt`.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    project=OUTPUT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    device=0,
    mosaic=0.5,
    mixup=0.0,
    fliplr=0.0,   # 좌우 flip 금지 (방향 정보)
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
)

## 5. 검증

In [ ]:
best_pt = os.path.join(OUTPUT_DIR, RUN_NAME, 'weights', 'best.pt')
print('best.pt:', best_pt, 'exists=', os.path.exists(best_pt))

best = YOLO(best_pt)
metrics = best.val(data=DATA_YAML, imgsz=IMGSZ, split='val')
print('mAP50   :', float(metrics.box.map50))
print('mAP50-95:', float(metrics.box.map))
print('names   :', best.names)

## 6. best.pt 다운로드

Drive 안 쓰므로 브라우저로 직접 다운로드. WSL에서 받아 `extract_labels.py --yolo_weights`로 사용.

In [ ]:
from google.colab import files
files.download(best_pt)

## 7. (선택) ONNX export

Jetson TRT engine용 중간 산출물. WSL standalone inference만이면 생략 가능.
TRT engine 빌드(trtexec)는 반드시 Jetson에서.

In [ ]:
best.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
onnx_path = best_pt.replace('.pt', '.onnx')
print('ONNX:', onnx_path, 'exists=', os.path.exists(onnx_path))
files.download(onnx_path)